In [ ]:
Author: Hui Fang

Purpose: ST 554 Project 2

Date: 3/15/2026

# Part I - Creating a Class

We are going to create our own class called SparkDataCheck that works on Spark SQL style data frames.

Create a .py file.

First import modules needed:

In [3]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from functools import reduce
from pyspark.sql.types import *
import pandas as pd
import numpy as np
import pyspark.pandas as ps

## Create a class Called SparkDataCheck

- Start a class called **SparkDataCheck**
- Create an `__init__` function that takes in self and a dataframe argument

  – Within this, create a `.df` attribute that is the dataframe

Let's start by creating our spark session

In [10]:
from pyspark.sql import SparkSession         # import the SparkSession class from PySpark#
spark = SparkSession.builder.getOrCreate()   # create or retrieve a SparkSession

Load a sample dataset for testing.

In [6]:
pdf = pd.read_csv("https://www4.stat.ncsu.edu/~online/datasets/red-wine.csv", delimiter = ";")
pdf.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


### Initiate a class and two classmethods

Initiate a class and create two @classmethods:

– One that creates an instance while reading in a csv file.\
   ∗ The method should have arguments for the class, the spark session, and the path to the file\
   ∗ You should use the `spark.read.load()` function as we did in our `pyspark` notebook. \
   ∗ Create an object of our class that is returned 
   
– One that creates an instance from a `pandas` dataframe (standard `pandas`)\
  ∗ The method should have arguments for the class, the spark session, and the pandas dataframe\
  ∗ You should use the `spark.CreateDataFrame()` function as we did in our `pyspark` notebook.\
  ∗ Create an object of our class that is returned

In [7]:
"""
SparkDataCheck.py

This module define a class that works 
on Spark SQL style data frames.
"""
# import modules needed
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from functools import reduce
from pyspark.sql.types import *
import pandas as pd
from pyspark.sql.types import NumericType
from pyspark.sql.functions import col as spark_col

class SparkDataCheck:
    def __init__(self, df: DataFrame):
        # create a .df attribute
        self.df = df
            
  #=================================================================
    # Classmethod 1: create instance by reading a CSV file
    @classmethod   
    def from_csv(cls, spark, path):
        df = (spark.read
                   .format("csv")
                   .option("header", True)
                   .option("inferSchema", True)
                   .option("sep", sep)
                   .load(path))
        return cls(df)
   
  
    #==============================================================
    # Classmethod 2: create instance from a pandas DataFrame
    @classmethod
    def from_pandas(cls,spark, pandas_df):
        df = spark.createDataFrame(pandas_df)
        return cls(df)
    
         
    

### Test the classmethods

In [5]:
import importlib
import ST554_project2_part1
importlib.reload(ST554_project2_part1)

<module 'ST554_project2_part1' from '/home/jupyter-hfang4@ncsu.edu/ST-554-Project-2/ST554_project2_part1.py'>

In [50]:
# check with the red-wine.csv data
check_df = ST554_project2_part1.SparkDataCheck.from_csv(spark, "red-wine.csv")
check_df.df.show(5)

+-------------+----------------+-----------+--------------+---------+-------------------+--------------------+-------+----+---------+-------+-------+
|fixed acidity|volatile acidity|citric acid|residual sugar|chlorides|free sulfur dioxide|total sulfur dioxide|density|  pH|sulphates|alcohol|quality|
+-------------+----------------+-----------+--------------+---------+-------------------+--------------------+-------+----+---------+-------+-------+
|          7.4|             0.7|        0.0|           1.9|    0.076|               11.0|                34.0| 0.9978|3.51|     0.56|    9.4|      5|
|          7.8|            0.88|        0.0|           2.6|    0.098|               25.0|                67.0| 0.9968| 3.2|     0.68|    9.8|      5|
|          7.8|            0.76|       0.04|           2.3|    0.092|               15.0|                54.0|  0.997|3.26|     0.65|    9.8|      5|
|         11.2|            0.28|       0.56|           1.9|    0.075|               17.0|           

In [12]:
pdf = pd.read_csv("red-wine.csv")
check_df2 = ST554_project2_part1.SparkDataCheck.from_pandas(spark, pdf)
check_df2.df.show(5)

+-------------+----------------+-----------+--------------+---------+-------------------+--------------------+-------+----+---------+-------+-------+
|fixed acidity|volatile acidity|citric acid|residual sugar|chlorides|free sulfur dioxide|total sulfur dioxide|density|  pH|sulphates|alcohol|quality|
+-------------+----------------+-----------+--------------+---------+-------------------+--------------------+-------+----+---------+-------+-------+
|          7.4|             0.7|        0.0|           1.9|    0.076|               11.0|                34.0| 0.9978|3.51|     0.56|    9.4|      5|
|          7.8|            0.88|        0.0|           2.6|    0.098|               25.0|                67.0| 0.9968| 3.2|     0.68|    9.8|      5|
|          7.8|            0.76|       0.04|           2.3|    0.092|               15.0|                54.0|  0.997|3.26|     0.65|    9.8|      5|
|         11.2|            0.28|       0.56|           1.9|    0.075|               17.0|           

This shows that the two classmethods are working. 

### Create validation methods

- Create a couple of validation methods. Each validation method will **modify the df attribute of the object** using different column functions and **return itself (the object that
has the df attribute)** so that we can chain commands. We won’t do any of the returning of the data within our methods (such as `take()` or `collect()`). We’ll leave that as something the user can do
themselves! (** This will need to be done via something like `my_object.df.show()`).

#### Create Boolean column based on numeric bounds

– Create a method that checks if each value in a numeric column is within user defined limits (upper and lower bounds, inclusive) and returns the dataframe with an appended column of
Boolean values.

∗ The function should allow the user to supply a single column and a `lower` and `upper` value. Check that at least one of lower or upper is provided (if not provided, don’t check that side).\
∗ For any `NULL` values, return `NULL`\
∗ If the user supplies a non-numeric column (not float, int, longint, bigint, double, or integer), print a message and return the df without modification.\
∗ Hints: Check out the `.dtypes` attribute of the data frame. On a column, you can use the `.between()` method

In [8]:
#============================================
# 1. Validation methods
#============================================

# 1.1 create boolean column based on numeric bounds

def check_numeric_range(self, col: str, lower: float = None, upper: float = None):
    """
    Append a Boolean column indicating whether values in a numeric column
    fall within user-defined lower and/or upper bounds (inclusive).
    NULL values remain NULL.
    Modifies self.df and returns self for method chaining.
    """
    # -----------------------------------
    # check if the column exists
    # -----------------------------------
    if col not in self.df.columns:
        print(f"Column '{col}' does not exist.")
        return self

    # -----------------------------------
    # check if the columin is numeric
    #------------------------------------
    dtype = self.df.schema[col].dataType
    if not isinstance(dtype, NumericType):
        print(f"Column '{col}' is not numeric.")
        return self

    #--------------------------------------
    # ensure at least one bound is provided
    #--------------------------------------
    if lower is None and upper is None:
        print("No bounds provided. Please provide at least one bound.")
        return self

    #----------------------------------
    # build the Boolean condition   
    #----------------------------------
    if lower is not None and upper is not None:
        # use Spark's between() when both bounds exist
        condition = spark_col(f"`{col}`").between(lower, upper)
    elif lower is not None:
        # only lower bound provided
        condition = spark_col(f"`{col}`") >= lower
    elif upper is not None:
        # only upper bound provided
        condition = spark_col(f"`{col}`") <= upper

    #---------------------------------
    # append Boolean column to dataframe
    #---------------------------------
    new_col_name = f"{col}_in_range"
    self.df = self.df.withColumn(new_col_name, condition)
    return self
    

#### Create a method checking string columns

– Create a method that checks if each value in a string column falls within a user specified set of levels and returns the dataframe with an appended column of Boolean values.
∗ For any `NULL` values, return `NUL`\
∗ If the user supplies a non-string column print a message and return the df without modification\
∗ Hint: The `.isin()` method on a column is useful!\

In [21]:
# 1.2 create a method checking values fall within a set of levels

def check_value_levels(self, col: str, levels):
    """
    Check whether values in a string column fall within 
    a user-specified set of allowed levels. 
    Appends a Boolean column. NULL values remain NULL.
    Modifies self.df and returns self for method chaining. 
    """
    # -----------------------------------
    # check if the column exists
    # -----------------------------------
    if col not in self.df.columns:
        print(f"Column '{col}' does not exist.")
        return self
    
    # -----------------------------------
    # check if the columin is string type
    #------------------------------------
    dtype = self.df.schema[col].dataType
    if not isinstance(dtype, StringType):
        print(f"Column '{col}' is not a string column.")
        return self
      
    # build the Boolean condition
    condition = spark_col(col).isin(levels)
    
    # append the new Boolean column
    new_col_name = f"{col}_in_levels"
    self.df = self.df.withColumn(new_col_name, condition)
    return self
        
    

#### Create a method checking missiing values

– Create a method that checks if a each value in a column is missing (`NULL` specifically) and returns the dataframe with an appended column of Boolean values.
∗ Hint: The `.isNULL()` method on a column is useful!

In [38]:
# 1.3 create a method that checks if each value in a column is missing

def check_value_missing(self, col: str):
    """
    Check whether values in a given column are NULL.
    Appends a Boolean column indicating NULL status.
    Modifies self.df and returns self for method chaining.
    """
    # -----------------------------------
    # check if the column exists
    # -----------------------------------
    if col not in self.df.columns:
        print(f"Column '{col}' does not exist.")
        return self
    
    # build the Bollean contion
    # .isNull() returns True, False, or NULL
    condition = self.spark_col(f"`{col}`").isNull()
    
    # append the new Boolean column to the dataframe
    new_col_name = f"{col}_is_null"
    self.df = self.df.withColumn(new_col_name, condition)
    
    return self
    

### Create summarization methods

- Create a couple of summarization methods (this will generally be writing our own way to use functions that exist!). Each summarization method will return the summarizations
of the data (not an amended version of the dataframe) as a `pandas` data frame (regular `pandas` not `pandas-on-spark`).

#### Create a method to report min and max of a variable

– Create a method to report the `min` and `max` of a numeric column supplied by the user. Add an optional grouping variable (only one grouping variable allowed for simplicity).\
  ∗ The method should check if the column is numeric. If so, it should report the min and max of the column (grouped if appropriate). If not, a message should be printed that the column isn’t numeric and `None` should be returned\
  ∗ If no column is supplied, the method should report the `min` and `max` of any numeric columns (and produce no messages otherwise), grouped if appropriate\
  ∗ Hints: This part got a bit complicated but I used the min and max functions from `pyspark.sql.functions` and the `.agg()` method on a `.groupBy()` spark SQL style data frame (similar to the notes). For the grouped option with all numeric columns, I used `reduce()` from `functools` with `pd.merge()` to simplify the result into a single data frame.

In [3]:
#====================================
# 2. Summarization methods
#====================================

# -----------------------------------------------------------------------------------
# 2.1 define a method to report min and max of a numeric coulumn supplied by the user
# -----------------------------------------------------------------------------------
# import module needed
from pyspark.sql.functions import min, max  

def min_max(self, col = None, group = None):
    """
    Report min and max for:
    1. a user-supplied numeric column (grouped if provided)
    2. or all numeric columns if no column is supplied (grouped if provided)
    Returns a pandas DataFrame
    """
    #-------------------------------------
    # scenario 1: User supplies a column
    #-------------------------------------
    if col is not None:
        # check column exists
        if col not in self.df.columns:
            print(f"Column {col} don't exist.")
            return None
        
        # get the data type of the column
        dtype = self.df.schema[col].dataType

        # check if column is numeric
        if not isinstance(dtype, NumericType):
            print(f"Column '{col}' is not numeric.")
            return None
        
        # check group exists if provided
        if group is not None and group not in self.df.columns:
            print(f"Group column '{group}' does not exist.")
            return None
        
        # grouped version for a single numeric column
        if group is not None:
            return(
                self.df.groupBy(group)
                       .agg(self.min(col).alias(f"{col}_min"),
                            self.max(col).alias(f"{col}_max"))
                       .toPandas()
            )
        
        # ungrouped version for a single numeric column      
        return (
            self.df.select(
                self.min(col).alias(f"{col}_min"),
                self.max(col).alias(f"{col}_max")
            ).toPandas()
        )
    
    #----------------------------------------------------------------
    # scenario 2: no column supplied: compute for all numeric columns
    #----------------------------------------------------------------
    # identify all numeric columns
    numeric_cols = [
        field.name
        for field in self.df.schema.fields
        if isinstance(field.dataType, NumericType)]
    
    # if no numeric columns exist, print a message
    if not numeric_cols:
        print("No numeric columns found.")
        return None
    
     # check group exists if provided
    if group is not None and group not in self.df.columns:
        print(f"Group column '{group}' does not exist.")
        return None
    
    # grouped version for all numeric columns
    if group is not None:
        # setup an enpty list
        dfs = []
        # compute grouped min/max for each numeric column 
        for c in numeric_cols:
            df_c = (
                self.df.groupBy(group)
                       .agg(self.min(c).alias(f"{c}_min"),
                            self.max(c).alias(f"{c}_max"))
            ).toPandas()
            dfs.append(df_c)
            
        # merge all grouped results into on DataFrame
        merged = reduce(lambda left, right: pd.merge(left, right, on = group),dfs)
        return merged
    
    # ungrouped scenartion for all numeric columns
    agg_exprs = []
    for c in numeric_cols:
        agg_exprs.extend([
            self.min(c).alisa(f"{c}_min"),
            self.max(c).alisa(f"{c}_max")
        ])
    # Return a DataFrame with all min/max values
    return self.df.select(*agg_exprs).toPandas()

#### Create a method to report counts of string columns

– Create a method to report the counts associated with one or two string columns. Have the function take in two separate arguments for columns, with the second being optional and the first
required.\
    ∗ The method should check if the column(s) are strings. If so, it should report the counts for the combinations of levels of each variable or of the single variable. If not, a message should
be printed that the column is numeric.

In [4]:
# 2.2 create a method that reports counts of one of two string columns
from pyspark.sql.functions import col as spark_col

def count_column(self, col1: str, col2: str = None):
    """
    Report counts associated with one or two string columns.
    The first column is required; the second is optional.
    Only string columns are allowed. If a column is not string,
    a message is printed and no counts are reported.
    """
    # -----------------------------------
    # check if column1 exists
    # -----------------------------------
    if col1 not in self.df.columns:
        print(f"Column '{col1}' does not exist.")
        return self
    
    # -----------------------------------------
    # if column is provided, check if it exists
    # -----------------------------------------
    if col2 is not None and col2 not in self.df.columns:
        print(f"Column '{col2}' does not exist. Counting only '{col1}'.")
        col2 = None   # force fallback to one‑column case
    
    # -----------------------------------
    # check if columin1 is string
    #------------------------------------
    dtype1 = self.df.schema[col1].dataType
    if not isinstance(dtype1, StringType):
        print(f"Column '{col1}' is not a string column.")
        return self
    
    # ----------------------------------------------
    # if column2 is provided, check if it is string
    #-----------------------------------------------
    if col2 is not None:
        dtype2 = self.df.schema[col2].dataType
        if not isinstance(dtype2, StringType):
            print(f"Column '{col2}' is not a string column. Counting only '{col1}'.")
            col2 = None   # force fallback to one‑column case
        
    #-----------------------------------
    # one-column case
    #-----------------------------------
    if col2 is None:
        print(f"Counts for '{col1}':")
        self.df.groupBy(col1).count().show()
        return self
    
    #----------------------------------------------
    # two-column case: both col1 and col2 provided
    #---------------------------------------------
    print(f"Counts for columns '{col1}' and '{col2}':")
    self.df.groupBy(col1, col2).count().show()
    
    return self    
    

## Check the created class with real data

Now we’ll use your class on some data! Create a .ipynb on the JupyterHub (use this same file for the steps
below and part II)
- In the notebook, provide an introduction and narrative to what you are about to do!
- Import your script so you have access to your class!
- Read in the air quality data we used in the first project. I’ve downloaded this data as a .csv file
and it is available at https://www4.stat.ncsu.edu/online/datasets/air.csv. Use your method
that creates an instance of the class from this csv file.
- Provide 4-5 examples of using each of your methods on this object. Show some examples where the
messages need to print out, where only one bound is provided, etc.
- Now, read that same data set in using pandas (not pandas-on-spark). Use your method to create an
instance of this class from the pandas data frame.
- Provide 1 example method call on that object.

### Instroduction

I developed a custom Python class, **SparkDataCheck**, which includes two `classmethods` for creating class instances, three validation methods that append Boolean columns to a Spark DataFrame, and two summarization methods that return pandas DataFrames. To demonstrate that the class works correctly on real data, I will use the [Air Quality dataset](https://archive.ics.uci.edu/dataset/360/air+quality) from the UCI Machine Learning Repository.

I begin by importing my Python script and reading the downloaded air quality CSV file. Using my `from_csv` classmethod, I create a `SparkDataCheck` object and apply each of my methods to this dataset. For each method, I provide four to five examples, including cases where the method prints warning messages (e.g., when a column does not exist or is the wrong type).

Next, I read the same dataset using pandas and use my `from_pandas` classmethod to create a second instance of the class. I then demonstrate one method on this pandas‑based object to confirm that both classmethods work as intended.

### Import my python script and clear up data

In [15]:
# Import my python script
from ST554_project2_part1 import SparkDataCheck

from pyspark.sql import SparkSession

spark = (SparkSession.builder.appName("my_app").getOrCreate())
# create object
air = SparkDataCheck.from_csv(spark, "air.csv", sep = ",")

Check data schema

In [4]:
air.df.printSchema()

root
 |-- _c0: integer (nullable = true)
 |-- Date: string (nullable = true)
 |-- Time: timestamp (nullable = true)
 |-- CO(GT): double (nullable = true)
 |-- PT08.S1(CO): integer (nullable = true)
 |-- NMHC(GT): integer (nullable = true)
 |-- C6H6(GT): double (nullable = true)
 |-- PT08.S2(NMHC): integer (nullable = true)
 |-- NOx(GT): integer (nullable = true)
 |-- PT08.S3(NOx): integer (nullable = true)
 |-- NO2(GT): integer (nullable = true)
 |-- PT08.S4(NO2): integer (nullable = true)
 |-- PT08.S5(O3): integer (nullable = true)
 |-- T: double (nullable = true)
 |-- RH: double (nullable = true)
 |-- AH: double (nullable = true)



Replace missing values with NULL, and create a categrical variable for test created method.

After replacing all missing values (coded as –200) with NULL,I renamed several sensor variables (PT08.S1(CO), PT08.S2(NMHC), PT08.S3(NOx), PT08.S4(NO₂), and PT08.S5(O₃)) to shorter, more intuitive names. I also dropped the `_c0` column since it not a variable. I then created a new categorical string variable to test the validation methods (such as `check_value_levels`). Specifically, I derived a variable called `Ben` from the numeric column `C6H6(GT)` by grouping its values into four categories based on the distribution of the data:
- < 4.5 -> "Low"
- 4.5 - 8 -> "Medium"
- 8 - 14 -> "High"
- 14 -> "Very high"

In [16]:
from pyspark.sql.functions import col, when
from pyspark.sql import functions as F

# replace missing values with None
for c, dtype in air.df.dtypes:
    if dtype in ("int", "double", "float", "bigint"):
        air.df = air.df.withColumn(c, when(col(f"`{c}`") == -200, None).otherwise(col(f"`{c}`")))

# rename variables 
air.df = (air.df.withColumnRenamed("PT08.S1(CO)", "CO")
                .withColumnRenamed("PT08.S2(NMHC)", "NMHC")
                .withColumnRenamed("PT08.S3(NOx)", "NOx")
                .withColumnRenamed("PT08.S4(NO2)", "NO2")
                .withColumnRenamed("PT08.S5(O3)", "O3")
         )

# drop index column
air.df = air.df.drop("_c0")
        
# create a new categorical variable "Ben"
air.df = air.df.withColumn("Ben",
                          when(col("C6H6(GT)") < 4.5, "Low")
                          .when(col("C6H6(GT)") < 8, "Medium")
                          .when(col("C6H6(GT)") < 14, "High")
                          .otherwise("Very high")
                          )

# show the first 10 rows of the dataset
air.df.show(10)

+---------+-------------------+------+----+--------+--------+----+-------+----+-------+----+----+----+----+------+------+
|     Date|               Time|CO(GT)|  CO|NMHC(GT)|C6H6(GT)|NMHC|NOx(GT)| NOx|NO2(GT)| NO2|  O3|   T|  RH|    AH|   Ben|
+---------+-------------------+------+----+--------+--------+----+-------+----+-------+----+----+----+----+------+------+
|3/10/2004|2026-03-19 18:00:00|   2.6|1360|     150|    11.9|1046|    166|1056|    113|1692|1268|13.6|48.9|0.7578|  High|
|3/10/2004|2026-03-19 19:00:00|   2.0|1292|     112|     9.4| 955|    103|1174|     92|1559| 972|13.3|47.7|0.7255|  High|
|3/10/2004|2026-03-19 20:00:00|   2.2|1402|      88|     9.0| 939|    131|1140|    114|1555|1074|11.9|54.0|0.7502|  High|
|3/10/2004|2026-03-19 21:00:00|   2.2|1376|      80|     9.2| 948|    172|1092|    122|1584|1203|11.0|60.0|0.7867|  High|
|3/10/2004|2026-03-19 22:00:00|   1.6|1272|      51|     6.5| 836|    131|1205|    116|1490|1110|11.2|59.6|0.7888|Medium|
|3/10/2004|2026-03-19 23

### Validation methods

#### Method 1: check numeric_range

In [20]:
# example 1: check a numeric variable and its range are provided
air.check_numeric_range("CO(GT)", 1232, 2654)

This output means that the method executed with no error. It returned `self`, which is my `SparkDataCheck` instance, and the memory address is 0x7f936e9c7850.

In [43]:
# example 2: check a non-exist variable
air.check_numeric_range("CO2", 20, 500)

Column 'CO2' does not exist.


The output means that the method executed without error. It also printed the message `"Column 'CO2' does not exist."` as expected.

In [30]:
# example 3: check a non-numeric variable is provided
air.check_numeric_range("Date", 1, 50)

Column 'Date' is not numeric.


The output means that the method executed without error. It also printed the message "Column 'Date' is not numeric." as expected.

In [34]:
# example 4: check a numeric variable and only the lower bound are provided
air.check_numeric_range("NMHC(GT)", 1000)

The output shows that the method executed without error. Because I provided only one numeric value, the method treated it as the lower bound by default and completed successfully.

In [11]:
# example 5: check a numeric variable and only the upper bound are provided
air.check_numeric_range("CO", upper = 1000)

The output shows that the method executed without error. Because I provided only the upper numeric value, the method completed successfully.

#### Method 2: check value levels in string columns

In [23]:
# example 1: check a non-exist variable and its level are provide
air.check_value_levels("Music", ["Good"])

Column 'Music' does not exist.


The output shows that the method executed without error. It detected that the column "Music" does not exist and printed out the message as expected.

In [24]:
# example 2: check a numeric variable and its level are provided
air.check_value_levels("NO2(GT)", "High")

Column 'NO2(GT)' is not a string column.


The output shows that the method executed without error. It detected the input column is not a string type and printed out the message as expected.

In [10]:
# example 3: check another numeric variable and its level are provided
air.check_value_levels("NO2", "Low")

Column 'NO2' is not a string column.


The output shows that the method executed without error. It detected the input column is not a string type and printed out the message as expected.

In [26]:
# example 4: check a string column and its levels are provided
air.check_value_levels("Ben", ["Low", "Medium"])
air.df.select("Ben", "Ben_in_levels").show(10)

+------+-------------+
|   Ben|Ben_in_levels|
+------+-------------+
|  High|        false|
|  High|        false|
|  High|        false|
|  High|        false|
|Medium|         true|
|Medium|         true|
|   Low|         true|
|   Low|         true|
|   Low|         true|
|   Low|         true|
+------+-------------+
only showing top 10 rows


The output shows that the method executed without error. The `check_value_levels` method creates a Boolean indicator showing whether each value in the `Ben` column matches one of the user‑specified allowed levels. Because the method performs a literal string comparison, only rows where the `Ben` value exactly equals one of the supplied strings (e.g., `"Low", "Medium"`) are marked `true` in the new column called `Ben_in_levels`. Rows whose values don't match the allowed list are marked `false`.

#### Method 3: check value missing

In [65]:
# example 1: check a non-exist variable is provided
air.check_value_missing("Benzene")

Column 'Benzene' does not exist.


The output shows that the method executed without error. It detected the input column `Benzene` does not exist and printed out the message as expected.

In [102]:
# example 2: check a string colunm is provided
air.check_value_missing("Ben")
air.df.select("Ben", "Ben_is_null").show(5)

+------+-----------+
|   Ben|Ben_is_null|
+------+-----------+
|  High|      false|
|  High|      false|
|  High|      false|
|  High|      false|
|Medium|      false|
+------+-----------+
only showing top 5 rows


The output indicates that the method executed successfully. The `check_value_missing` method added a new Boolean column (`Ben_is_null`) showing whether each value in the `Ben` column is **NULL**. Because the `Ben` variable was created from `C6H6(GT)` after missing values were already replaced, none of its values are NULL, so all rows are marked `false`.

In [103]:
# example 3: check two numeric veriable is provided
air.check_value_missing("RH")
air.df.select("RH", "RH_is_null").show(5)

+----+----------+
|  RH|RH_is_null|
+----+----------+
|48.9|     false|
|47.7|     false|
|54.0|     false|
|60.0|     false|
|59.6|     false|
+----+----------+
only showing top 5 rows


The output indicates that the method executed successfully. The check_value_missing method added a new Boolean column (`RH_is_null`) showing whether each value in the `RH` column is `NULL`. Because missing values of the `RH` variable were already replaced, none of its values are NULL, so all rows are marked `false`.

In [106]:
# example 4: check another numeric variable is provided
air.check_value_missing("NOx(GT)")
air.df.select("NOx(GT)", "NOx(GT)_is_null").show(5)

+-------+---------------+
|NOx(GT)|NOx(GT)_is_null|
+-------+---------------+
|    166|          false|
|    103|          false|
|    131|          false|
|    172|          false|
|    131|          false|
+-------+---------------+
only showing top 5 rows


The output indicates that the method executed successfully. The check_value_missing method added a new Boolean column (`NOx(GT)_is_null`) showing whether each value in the `NOx(GT)` column is `NULL`. Because missing values of the `NOx(GT)` variable were already replaced, none of its values are NULL, so all rows are marked `false`.

### Summarization methods

#### Method 1: check min and max of a numeric variable

In [13]:
# example 1: check one numeric variable
air.min_max("NMHC")

,NMHC_min,NMHC_max
0,383,2214


The output shows that the method executed successfully. For the variable `NMHC`, the minimum value is 383 and the maximum value is 2214.

In [14]:
# example 2: check a numeric variable grouped by a string variable levels
air.min_max("CO", "Ben")

,Ben,CO_min,CO_max
0,High,846,1586
1,Low,647,1185
2,Medium,784,1305
3,Very high,966,2040


The output shows that the method executed successfully. For the variable `CO`, grouped by the categories of `Ben`, the minimum and maximum values for each level are displayed.

In [10]:
# example 3: check no variable provided, the method computes min and max for all numerical variables
# transpose the output to vertical
air.min_max().T

,0
CO(GT)_min,0.1000
CO(GT)_max,11.9000
CO_min,647.0000
CO_max,2040.0000
NMHC(GT)_min,7.0000
NMHC(GT)_max,1189.0000
C6H6(GT)_min,0.1000
C6H6(GT)_max,63.7000
NMHC_min,383.0000
NMHC_max,2214.0000


The output shows the method executed successfully. When no variable is provided, the method calculated and displayed the minimum and maximum values for all numeric variables. 

In [36]:
# example 4: check a string variable is provided, the method will display a message
air.min_max("Ben")

Column 'Ben' is not numeric.


When the string variable `Ben` is provided, the method correctly detects that it is not numeric and displays the expected message.

In [11]:
# example 5: check only string variable is provided, the method will compute min and max grouped by the levels of the string variable
air.min_max(group = "Ben").T

,0,1,2,3
Ben,High,Low,Medium,Very high
CO(GT)_min,0.1,0.1,0.1,0.3
CO(GT)_max,5.5,2.8,4.3,11.9
CO_min,846,647,784,966
CO_max,1586,1185,1305,2040
NMHC(GT)_min,64,7,38,66
NMHC(GT)_max,451,116,252,1189
C6H6(GT)_min,8.0,0.1,4.5,14.0
C6H6(GT)_max,13.9,4.4,7.9,63.7
NMHC_min,897,383,735,1116


The output shows that when no variable is provided but the grouping variable `Ben` is supplied, the method computes and displays the minimum and maximum values for each numeric variable, grouped by each level of `Ben`. The method executed without error.

#### Method 2: count string columns

In [39]:
# example 1: check a string conlumn is provided 
air.count_column("Ben")

Counts for 'Ben':
+---------+-----+
|      Ben|count|
+---------+-----+
|     High| 2389|
|      Low| 2255|
|   Medium| 2091|
|Very high| 2622|
+---------+-----+



The output shows that the method executed successfully. The method correctly computes and displays the counts across the levels of the string column `Ben`.

In [17]:
# example 2: check if a numeric column is provided, the method will dispaly message.
air.count_column("NOx")

Column 'NOx' is not a string column.


The output shows that the method executed successfully. The method correctly detected that `NOx` is not a string column and displayed the expected message.

In [18]:
# example 3: check two string columns are provided
air.count_column("Ben", "Date")

Counts for columns 'Ben' and 'Date':
+---------+----------+-----+
|      Ben|      Date|count|
+---------+----------+-----+
|   Medium| 9/26/2004|    8|
|     High| 10/2/2004|   16|
|     High|10/29/2004|    5|
|Very high| 11/8/2004|    4|
|   Medium|  3/2/2005|    5|
|      Low|  5/9/2004|    2|
|     High| 5/20/2004|   10|
|   Medium| 9/21/2004|    1|
|   Medium| 11/9/2004|    4|
|   Medium|11/13/2004|    5|
|   Medium|  2/3/2005|    2|
|      Low|  5/6/2004|    6|
|Very high| 7/19/2004|   12|
|      Low| 8/29/2004|    7|
|Very high|12/19/2004|    5|
|      Low|12/21/2004|    8|
|     High|  1/9/2005|    8|
|   Medium| 1/11/2005|    3|
|   Medium| 1/16/2005|   10|
|   Medium|  4/4/2004|    9|
+---------+----------+-----+
only showing top 20 rows


The output shows that the method executed successfully. The method grouped the data by `Ben` and `Date`, computed the counts for each combination, and displayed the results.

In [19]:
# example 4: check a string column and a numeric column are provided
air.count_column("Ben", "NO2")

Column 'NO2' is not a string column. Counting only 'Ben'.
Counts for 'Ben':
+---------+-----+
|      Ben|count|
+---------+-----+
|     High| 2389|
|      Low| 2255|
|   Medium| 2091|
|Very high| 2622|
+---------+-----+



The output shows that the method exectued successfully. The method corrected detected that 'NO2' is not a string column, displayed expected message, and still reported the counts for each level of the string column `Ben`.

In [23]:
# example 5: check a string column and a non-exist column are provided
air.count_column("Ben", "CO2")

Column 'CO2' does not exist. Counting only 'Ben'.
Counts for 'Ben':
+---------+-----+
|      Ben|count|
+---------+-----+
|     High| 2389|
|      Low| 2255|
|   Medium| 2091|
|Very high| 2622|
+---------+-----+



The output shows that the method exectued successfully. The method corrected detected that 'CO2' does not exist, displayed expected message, and still reported the ounts for each level of the string column `Ben`.

## Check the data set with `pandas`

Now, read that same data set in using `pandas` (not `pandas-on-spark`). Use my method to create an instance of this class from the pandas data frame.\
• Provide 1 example method call on that object.

#### Read in data and check its structure

In [33]:
import pandas as pd
pdf = pd.read_csv("air.csv")
pdf.head()


,Unnamed: 0,Date,Time,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH
0,0,3/10/2004,18:00:00,2.6,1360,150,11.9,1046,166,1056,113,1692,1268,13.6,48.9,0.7578
1,1,3/10/2004,19:00:00,2.0,1292,112,9.4,955,103,1174,92,1559,972,13.3,47.7,0.7255
2,2,3/10/2004,20:00:00,2.2,1402,88,9.0,939,131,1140,114,1555,1074,11.9,54.0,0.7502
3,3,3/10/2004,21:00:00,2.2,1376,80,9.2,948,172,1092,122,1584,1203,11.0,60.0,0.7867
4,4,3/10/2004,22:00:00,1.6,1272,51,6.5,836,131,1205,116,1490,1110,11.2,59.6,0.7888


#### Clean data, rename variables, and create a categorical variable

As in the earlier steps, I replace missing values coded as `-200` with `None`, removed the index column, renamed the sensor varibles, and created the categorical variable `Benzene` from `C6H6(GT)`.

In [40]:
# clean the pandas DataFrame
pdf = pdf.replace(-200, None) # replace missing values
pdf = pdf.drop(columns = ["_c0", "Unnamed: 0"], errors="ignore") # drop the index column
# Rename variables
pdf = pdf.rename(columns = {"PT08.S1(CO)": "S1_CO",
                            "PT08.S2(NMHC)": "S2_NMHC",
                            "PT08.S3(NOx)": "S3_NOx",
                            "PT08.S4(NO2)": "S4_NO2",
                            "PT08.S5(O3)": "S5_O3"})

# create a categrogical variable Ben
pdf["Benzene"] = pd.cut(pdf["C6H6(GT)"],
    bins = [-float("inf"), 4.5, 8, 14, float("inf")],
    labels = ["Low", "Medium", "High", "Very high"])

# Convert to SparkDataCheck
air_pd = SparkDataCheck.from_pandas(spark,pdf)
# check the data frame
air_pd.df.show(10)

+---------+--------+------+-----+--------+--------+-------+-------+------+-------+------+-----+----+----+------+-------+
|     Date|    Time|CO(GT)|S1_CO|NMHC(GT)|C6H6(GT)|S2_NMHC|NOx(GT)|S3_NOx|NO2(GT)|S4_NO2|S5_O3|   T|  RH|    AH|Benzene|
+---------+--------+------+-----+--------+--------+-------+-------+------+-------+------+-----+----+----+------+-------+
|3/10/2004|18:00:00|   2.6| 1360|     150|    11.9|   1046|    166|  1056|    113|  1692| 1268|13.6|48.9|0.7578|   High|
|3/10/2004|19:00:00|   2.0| 1292|     112|     9.4|    955|    103|  1174|     92|  1559|  972|13.3|47.7|0.7255|   High|
|3/10/2004|20:00:00|   2.2| 1402|      88|     9.0|    939|    131|  1140|    114|  1555| 1074|11.9|54.0|0.7502|   High|
|3/10/2004|21:00:00|   2.2| 1376|      80|     9.2|    948|    172|  1092|    122|  1584| 1203|11.0|60.0|0.7867|   High|
|3/10/2004|22:00:00|   1.6| 1272|      51|     6.5|    836|    131|  1205|    116|  1490| 1110|11.2|59.6|0.7888| Medium|
|3/10/2004|23:00:00|   1.2| 1197

#### Check one method

In [43]:
# one example method call for min and max 
air_pd.min_max("S1_CO")

,S1_CO_min,S1_CO_max
0,647,2040


The output shows that the method exectued successfully. The method computed and displayed the min and max of variable `S1_CO`.

## Brief summary of Part I

In Part I of the project, I developed a class named `SparkDataCheck`, which includes two `@classmethod`s, three validation methods, and two summarization methods. I read and cleaned up the Air Quality dataset from the UCI Machine Learning Repository and used it to test each method with several examples. All methods executed successfully, producing the expected values or messages. I also loaded the same dataset using `pandas`, created a `SparkDataCheck` instance from the `pandas` DataFrame, and verified that the min_max method ran without errors, confirming that the class was constructed correctly.

In [13]:

import importlib
import ST554_project2_part1
importlib.reload(ST554_project2_part1)


<module 'ST554_project2_part1' from '/home/jupyter-hfang4@ncsu.edu/ST-554-Project-2/ST554_project2_part1.py'>

# Part II - Basic spark analysis

### Introduction

In Part II, we use `pandas`‑on‑Spark and Spark SQL to perform a brief exploratory analysis of NFL weekly data. After loading and inspecting the dataset, we focus on quarterback (QB) statistics from the 2005–2023 regular seasons, compute season‑level summaries for each player, create two derived performance metrics, and rank players based on these measures. We then repeat the workflow using Spark SQL to compare results across the two APIs. The specific steps and instructions are outlined below.


- Read in the weekly nfl data (csv file is available at the project page and needs to upload it to JupyterHub)
- Check out the first 5 rows of the `DataFrame`
- Report all of the column names
- We want to only look at QB stats for the seasons 2005 to 2023 (inclusive).\
    – Subset the rows of the data to only include the position “QB”, the regular season (“REG”), and season in the range noted above\
    – Subset the columns to only include the `player_display_name`, `season`, `week`, `completions`, `attempts`, `passing_yards`, `passing_tds`, and `interceptions`\
    – For each `player_display_name` and `season` combination, fine the _sum_ and *mean* of each of the statistical quantities (the rest of the columns we chose above)\
    – Create two new variables (by season/player combination):\
        ∗ `completion_percentage` = (sum of completions)/(sum of attempts)\
        ∗ `td_int_ratio` = (sum passing tds)/(sum interceptions)
- Save the result of above as an object. With that object
    – Subset the rows to only include player/season combinations wher ethe sum of attempts is at least 50.\
    – Sort the rows descending by `completion_percentage` and report the first 40 values!\
    – Sort the rows descending by `td_int_ratio` and report the first 40 values!
- Repeat the above completely using te Spark SQL DataFrame, including reading in the data!\
    – Note: the `td_int_ratio` values are treated differently between `pandas`-on-Spark and Spark SQL. Note this difference when it happens! 


### Using solely `pandas`-on-Spark

Let's first import modules needed.

In [18]:
import pandas as pd
import numpy as np
import pyspark.pandas as ps
# tells pandas‑on‑Spark to keep working even if Spark’s ANSI SQL mode is enabled.
ps.set_option('compute.fail_on_ansi_mode', False)


Then read in data.

In [19]:
# read in data
nfl = pd.read_csv("weekly_nfl_data.csv")                
# convert to pandas DataFrame
nfl_ps = ps.from_pandas(nfl)
# check the first 5 rows of the DataFrame
nfl_ps.head()

,player_id,player_name,player_display_name,position,position_group,headshot_url,recent_team,season,week,season_type,opponent_team,completions,attempts,passing_yards,passing_tds,interceptions,sacks,sack_yards,sack_fumbles,sack_fumbles_lost,passing_air_yards,passing_yards_after_catch,passing_first_downs,passing_epa,passing_2pt_conversions,pacr,dakota,carries,rushing_yards,rushing_tds,rushing_fumbles,rushing_fumbles_lost,rushing_first_downs,rushing_epa,rushing_2pt_conversions,receptions,targets,receiving_yards,receiving_tds,receiving_fumbles,receiving_fumbles_lost,receiving_air_yards,receiving_yards_after_catch,receiving_first_downs,receiving_epa,receiving_2pt_conversions,racr,target_share,air_yards_share,wopr,special_teams_tds,fantasy_points,fantasy_points_ppr
0,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,1,REG,DEN,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,16,60.0,1,0.0,0.0,4.0,6.248771,0,1,1,7.0,0,0.0,0.0,0.0,0.0,0.0,0.292378,0,0.0,0.052632,NaN,NaN,0.0,12.7,13.7
1,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,2,REG,ARI,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,9,33.0,0,0.0,0.0,1.0,-1.434950,0,3,4,18.0,0,0.0,0.0,0.0,0.0,1.0,0.377009,0,0.0,0.117647,NaN,NaN,0.0,5.1,8.1
2,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,4,REG,BUF,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,3,2.0,0,0.0,0.0,0.0,-1.539952,0,0,1,0.0,0,0.0,0.0,0.0,0.0,0.0,-0.699578,0,NaN,0.023810,NaN,NaN,0.0,0.2,0.2
3,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,CLE,1999,7,REG,LA,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,6,27.0,0,0.0,0.0,0.0,0.216051,0,2,2,8.0,0,0.0,0.0,0.0,0.0,0.0,-0.228454,0,0.0,0.050000,NaN,NaN,0.0,3.5,5.5
4,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,CLE,1999,8,REG,NO,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,13,39.0,0,0.0,0.0,2.0,-2.972259,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,NaN,0,NaN,NaN,NaN,NaN,0.0,3.9,3.9


Report all of the column names:

In [6]:
# Get all the column names
nfl_ps.columns

Index(['player_id', 'player_name', 'player_display_name', 'position',
       'position_group', 'headshot_url', 'recent_team', 'season', 'week',
       'season_type', 'opponent_team', 'completions', 'attempts',
       'passing_yards', 'passing_tds', 'interceptions', 'sacks', 'sack_yards',
       'sack_fumbles', 'sack_fumbles_lost', 'passing_air_yards',
       'passing_yards_after_catch', 'passing_first_downs', 'passing_epa',
       'passing_2pt_conversions', 'pacr', 'dakota', 'carries', 'rushing_yards',
       'rushing_tds', 'rushing_fumbles', 'rushing_fumbles_lost',
       'rushing_first_downs', 'rushing_epa', 'rushing_2pt_conversions',
       'receptions', 'targets', 'receiving_yards', 'receiving_tds',
       'receiving_fumbles', 'receiving_fumbles_lost', 'receiving_air_yards',
       'receiving_yards_after_catch', 'receiving_first_downs', 'receiving_epa',
       'receiving_2pt_conversions', 'racr', 'target_share', 'air_yards_share',
       'wopr', 'special_teams_tds', 'fantasy_points

- Subset the rows of the data to only include the position “QB”, the regular season (“REG”), and season 2005-2023
- Subset the columns to only include the `player_display_name`, `season`, `week`, `completions`, `attempts`, `passing_yards`, `passing_tds`, and `interceptions`

In [20]:
# subset the data based on conditions
qb = nfl_ps[
    (nfl_ps.position == "QB") & 
    (nfl_ps.season.between(2005, 2023)) &
    (nfl_ps.season_type == "REG")
]

qb_select = qb[["player_display_name", "season", "week",
     "completions", "attempts", "passing_yards",
     "passing_tds", "interceptions"]] 

# show the first 5 rows of subset data frame
qb_select.head()

,player_display_name,season,week,completions,attempts,passing_yards,passing_tds,interceptions
29406,Tony Banks,2005,17,14,25,173.0,1,2.0
29426,Charlie Batch,2005,9,9,16,65.0,0,1.0
29427,Charlie Batch,2005,10,13,19,150.0,0,0.0
29428,Charlie Batch,2005,16,1,1,31.0,1,0.0
29447,Jeff Blake,2005,2,1,1,11.0,0,0.0


- For each `player_display_name` and `season` combination, find the *sum* and *mean* of each of the statistical quantities (the rest of the columns we chose above)

In [21]:
qb_summary = (
    qb_select
    .groupby(["player_display_name", "season"])
    .agg({
        "week":["sum", "mean"],
        "completions": ["sum", "mean"],
        "attempts": ["sum", "mean"],
        "passing_yards": ["sum", "mean"],
        "passing_tds": ["sum", "mean"],
        "interceptions": ["sum", "mean"]
    })
)
# show the first 5 rows and round to 3 dicimals
qb_summary.head().round(3)

week         completions         attempts         passing_yards          passing_tds        interceptions       
                            sum    mean         sum    mean      sum    mean           sum     mean         sum   mean           sum   mean
player_display_name season                                                                                                                 
Jeff Blake          2005     19   9.500           8   4.000        9   4.500          55.0   27.500           1  0.500           0.0  0.000
Daunte Culpepper    2005     31   4.429         139  19.857      216  30.857        1564.0  223.429           6  0.857          12.0  1.714
Kerry Collins       2005    134   8.933         302  20.133      566  37.733        3759.0  250.600          20  1.333          13.0  0.867
Tony Banks          2005     17  17.000          14  14.000       25  25.000         173.0  173.000           1  1.000           2.0  2.000
Charlie Batch       2005     35  11.667          23   7.667       36  12.000         246.0   82.000           1  0.333           1.0  0.333

The output displays the first 5 rows of the grouped results, showing the `sum` and `mean` of `week`, `completions`, `attempts`, `passing_yards`, `passing_tds`, and `interceptions` for each `player_dispaly_name` and `season` combination.

- Create two new variables (by season/player combination):\
    ∗ completion_percentage = (sum of completions)/(sum of attempts)\
    ∗ td_int_ratio = (sum passing tds)/(sum interceptions)

Since the grouped `qb_summary` DataFrame has multiIndex column names (e.g., ('completions', 'sum') and ('completions', 'mean')), these need to be converted into flat names such as `completions_sum` and `completions_mean`.

In [22]:
# flatten the column names
qb_summary.columns = ["_".join(col).rstrip("_") for col in qb_summary.columns]

In [23]:
# check columns after flatting
qb_summary.columns

Index(['week_sum', 'week_mean', 'completions_sum', 'completions_mean',
       'attempts_sum', 'attempts_mean', 'passing_yards_sum',
       'passing_yards_mean', 'passing_tds_sum', 'passing_tds_mean',
       'interceptions_sum', 'interceptions_mean'],
      dtype='object')

In [24]:
# create new variables
qb_summary["completion_percentage"] = (qb_summary.completions_sum / qb_summary.attempts_sum.where(qb_summary.attempts_sum != 0))
qb_summary["td_int_ratio"] = (qb_summary.passing_tds_sum / qb_summary.interceptions_sum.where(qb_summary.interceptions_sum != 0))
# save the results to an oect
qb_new = qb_summary
# show the first 5 rows and round to 3 deicimals
qb_new.head().round(3)

,,week_sum,week_mean,completions_sum,completions_mean,attempts_sum,attempts_mean,passing_yards_sum,passing_yards_mean,passing_tds_sum,passing_tds_mean,interceptions_sum,interceptions_mean,completion_percentage,td_int_ratio
player_display_name,season,,,,,,,,,,,,,,
Jeff Blake,2005,19,9.500,8,4.000,9,4.500,55.0,27.500,1,0.500,0.0,0.000,0.889,NaN
Daunte Culpepper,2005,31,4.429,139,19.857,216,30.857,1564.0,223.429,6,0.857,12.0,1.714,0.644,0.500
Kerry Collins,2005,134,8.933,302,20.133,566,37.733,3759.0,250.600,20,1.333,13.0,0.867,0.534,1.538
Tony Banks,2005,17,17.000,14,14.000,25,25.000,173.0,173.000,1,1.000,2.0,2.000,0.560,0.500
Charlie Batch,2005,35,11.667,23,7.667,36,12.000,246.0,82.000,1,0.333,1.0,0.333,0.639,1.000


This output shows that the column names have been flattened and that the new variables `completion_percentage` and `td_int_ratio` were successfully created and appended to the right side of the DataFrame, with their corresponding values calculated.

Now, with that object qb_new, we will\
    – Subset the rows to only include player/season combinations where the sum of attempts is at least 50.\
    – Sort the rows descending by completion_percentage and report the first 40 values!\
    – Sort the rows descending by td_int_ratio and report the first 40 values!

In [25]:
# subset the rows to only include player/season combinations where the sum of attempts is at least 50.
qb_new_sub = qb_new.loc[qb_new.attempts_sum > 50]

# Sort the rows descending by completion_percentage and report the first 40 values!
qb_new_sub.sort_values("completion_percentage", ascending = False).head(40)

week_sum  week_mean  completions_sum  completions_mean  attempts_sum  attempts_mean  passing_yards_sum  passing_yards_mean  passing_tds_sum  passing_tds_mean  interceptions_sum  interceptions_mean  completion_percentage  td_int_ratio
player_display_name season                                                                                                                                                                                                                                           
C.J. Beathard       2023          65  10.833333               40          6.666667            53       8.833333              349.0           58.166667                1          0.166667                0.0            0.000000               0.754717           NaN
Colt McCoy          2021          62   8.857143               74         10.571429            99      14.142857              740.0          105.714286                3          0.428571                1.0            0.142857               0.747475      3.000000
Matt Schaub         2019          52  10.400000               50         10.000000            67      13.400000              580.0          116.000000                3          0.600000                1.0            0.200000               0.746269      3.000000
Drew Brees          2018         130   8.666667              364         24.266667           489      32.600000             3992.0          266.133333               32          2.133333                5.0            0.333333               0.744376      6.400000
                    2019         119  10.818182              281         25.545455           378      34.363636             2979.0          270.818182               27          2.454545                4.0            0.363636               0.743386      6.750000
Mason Rudolph       2023          66  16.500000               55         13.750000            74      18.500000              719.0          179.750000                3          0.750000                0.0            0.000000               0.743243           NaN
Taysom Hill         2020         147   9.187500               88          5.500000           121       7.562500              928.0           58.000000                4          0.250000                2.0            0.125000               0.727273      2.000000
Nick Foles          2018          51  10.200000              141         28.200000           195      39.000000             1413.0          282.600000                7          1.400000                4.0            0.800000               0.723077      1.750000
Drew Brees          2017         148   9.250000              386         24.125000           536      33.500000             4334.0          270.875000               23          1.437500                8.0            0.500000               0.720149      2.875000
Sam Bradford        2016         146   9.733333              395         26.333333           552      36.800000             3877.0          258.466667               20          1.333333                5.0            0.333333               0.715580      4.000000
Drew Brees          2011         142   8.875000              471         29.437500           660      41.250000             5535.0          345.937500               46          2.875000               14.0            0.875000               0.713636      3.285714
Colt McCoy          2014          57  11.400000               91         18.200000           128      25.600000             1057.0          211.400000                4          0.800000                3.0            0.600000               0.710938      1.333333
Aaron Rodgers       2020         148   9.250000              372         23.250000           526      32.875000             4299.0          268.687500               48          3.000000                5.0            0.312500               0.707224      9.600000
Bailey Zappe        2022          22   5.500000               65         16.250000            92  

The output shows that the DataFrame has been sorted descending by `completion_percentage` and the 40 values are dispalyed.

Now, let's sort the rows descending by `td_int_ratio` and report the first 40 values!

In [26]:
qb_new_sub.sort_values("td_int_ratio", ascending=False).head(40)

,,week_sum,week_mean,completions_sum,completions_mean,attempts_sum,attempts_mean,passing_yards_sum,passing_yards_mean,passing_tds_sum,passing_tds_mean,interceptions_sum,interceptions_mean,completion_percentage,td_int_ratio
player_display_name,season,,,,,,,,,,,,,,
Matt Schaub,2005,65,9.285714,33,4.714286,64,9.142857,495.0,70.714286,4,0.571429,0.0,0.000000,0.515625,NaN
Charlie Batch,2006,71,10.142857,30,4.285714,52,7.428571,477.0,68.142857,5,0.714286,0.0,0.000000,0.576923,NaN
Todd Collins,2007,62,15.500000,67,16.750000,105,26.250000,888.0,222.000000,5,1.250000,0.0,0.000000,0.638095,NaN
Kerry Collins,2007,48,8.000000,50,8.333333,82,13.666667,531.0,88.500000,0,0.000000,0.0,0.000000,0.609756,NaN
Troy Smith,2007,62,15.500000,40,10.000000,76,19.000000,452.0,113.000000,2,0.500000,0.0,0.000000,0.526316,NaN
Jake Locker,2011,51,10.200000,34,6.800000,66,13.200000,542.0,108.400000,4,0.800000,0.0,0.000000,0.515152,NaN
Derek Anderson,2014,30,6.000000,65,13.000000,97,19.400000,701.0,140.200000,5,1.000000,0.0,0.000000,0.670103,NaN
Brian Hoyer,2016,27,4.500000,134,22.333333,200,33.333333,1445.0,240.833333,6,1.000000,0.0,0.000000,0.670000,NaN
Nick Foles,2016,17,8.500000,36,18.000000,55,27.500000,410.0,205.000000,3,1.500000,0.0,0.000000,0.654545,NaN


The output shows the DataFrame sorted in descending order by `td_int_ratio`, and the first 40 values are displayed. Rows with `NaN` appear first because Spark treats missing values as larger than numeric values when sorting in descending order.

### Using the Spark SQL DataFrame

#### Import modules and read in data

In [2]:
# import SparkSession
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import pandas as pd

spark = (SparkSession.builder
    .appName("nfl_app")
    .config("spark.sql.ansi.enabled", "false")
    .getOrCreate()
)

- Read in data and check the first five rows.

In [4]:
# read in data
nfl_sdf = spark.read.csv("weekly_nfl_data.csv", header = True, inferSchema = True)

# check out the first 5 rows of the DataFrame
nfl_sdf.limit(5).toPandas()

,player_id,player_name,player_display_name,position,position_group,headshot_url,recent_team,season,week,season_type,...,receiving_first_downs,receiving_epa,receiving_2pt_conversions,racr,target_share,air_yards_share,wopr,special_teams_tds,fantasy_points,fantasy_points_ppr
0,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,1,REG,...,0.0,0.292378,0,0.0,0.052632,NaN,NaN,0.0,12.7,13.7
1,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,2,REG,...,1.0,0.377009,0,0.0,0.117647,NaN,NaN,0.0,5.1,8.1
2,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,4,REG,...,0.0,-0.699578,0,NaN,0.023810,NaN,NaN,0.0,0.2,0.2
3,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,CLE,1999,7,REG,...,0.0,-0.228454,0,0.0,0.050000,NaN,NaN,0.0,3.5,5.5
4,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,CLE,1999,8,REG,...,0.0,NaN,0,NaN,NaN,NaN,NaN,0.0,3.9,3.9


The output shows the first five rows of the Spark SQL DataFrame. Because `nfl_sdf.show(5)` truncates long columns and doesn’t display the table as clearly, I used `nfl_sdf.limit(5).toPandas()` to view the same rows in a more readable, pandas‑style format.

#### Report all the column names

In [6]:
nfl_sdf.columns

['player_id',
 'player_name',
 'player_display_name',
 'position',
 'position_group',
 'headshot_url',
 'recent_team',
 'season',
 'week',
 'season_type',
 'opponent_team',
 'completions',
 'attempts',
 'passing_yards',
 'passing_tds',
 'interceptions',
 'sacks',
 'sack_yards',
 'sack_fumbles',
 'sack_fumbles_lost',
 'passing_air_yards',
 'passing_yards_after_catch',
 'passing_first_downs',
 'passing_epa',
 'passing_2pt_conversions',
 'pacr',
 'dakota',
 'carries',
 'rushing_yards',
 'rushing_tds',
 'rushing_fumbles',
 'rushing_fumbles_lost',
 'rushing_first_downs',
 'rushing_epa',
 'rushing_2pt_conversions',
 'receptions',
 'targets',
 'receiving_yards',
 'receiving_tds',
 'receiving_fumbles',
 'receiving_fumbles_lost',
 'receiving_air_yards',
 'receiving_yards_after_catch',
 'receiving_first_downs',
 'receiving_epa',
 'receiving_2pt_conversions',
 'racr',
 'target_share',
 'air_yards_share',
 'wopr',
 'special_teams_tds',
 'fantasy_points',
 'fantasy_points_ppr']

#### Subset DataFrame

- Subset the rows of the data to only include the position "QB", the regular season("REG"), and season 2005-2023.
- Subset the columns to only include the `player_display_name`, `season`, `week`, `completions`, `attempts`, `passing_yards`, `passing_tds`, and `interceptions`.

In [6]:
# subset data to include "QB", season("REG"),and season 2005-2023
qb_sdf = nfl_sdf.filter((F.col("position") == "QB") &
    (F.col("season_type") == "REG") &
    (F.col("season").between(2005, 2023))
)

# select 8 columns of interest
qb_sdf_sub = qb_sdf.select("player_display_name", "season", "week",
    "completions", "attempts", "passing_yards", "passing_tds", "interceptions")

# dispaly the first 5 rows of selected columns
qb_sdf_sub.limit(5).toPandas()

,player_display_name,season,week,completions,attempts,passing_yards,passing_tds,interceptions
0,Tony Banks,2005,17,14,25,173.0,1,2.0
1,Charlie Batch,2005,9,9,16,65.0,0,1.0
2,Charlie Batch,2005,10,13,19,150.0,0,0.0
3,Charlie Batch,2005,16,1,1,31.0,1,0.0
4,Jeff Blake,2005,2,1,1,11.0,0,0.0


The oupput shows that the DataFrame was successfully subset according to the filtering criteria and the first five rows of the eight selected columns are dispalyed. 

#### Compute sum and mean

- For each `player_display_name` and `season` combination, fine the sum and mean of each of the statistical quantities

In [39]:
# compute the sum and mean grouped be player_display_name and season
qb_sub_summary = qb_sdf_sub.groupBy("player_display_name", "season").agg(F.sum("season").alias("season_sum"),
         F.mean("season").alias("season_mean"),
         F.sum("completions").alias("completions_sum"),
         F.mean("completions").alias("completions_mean"),
         F.sum("attempts").alias("attempts_sum"),
         F.mean("attempts").alias("attempts_mean"),
         F.sum("passing_yards").alias("passing_yards_sum"),
         F.mean("passing_yards").alias("passing_yards_mean"),
         F.sum("passing_tds").alias("passing_tds_sum"),
         F.mean("passing_yards").alias("passing_yards_mean"),
         F.sum("interceptions").alias("interceptions_sum"),
         F.mean("interceptions").alias("interceptions_mean"))
# show the top 5 rows
qb_sub_summary.show(5)

+-------------------+------+----------+-----------+---------------+------------------+------------+------------------+-----------------+------------------+---------------+------------------+-----------------+------------------+
|player_display_name|season|season_sum|season_mean|completions_sum|  completions_mean|attempts_sum|     attempts_mean|passing_yards_sum|passing_yards_mean|passing_tds_sum|passing_yards_mean|interceptions_sum|interceptions_mean|
+-------------------+------+----------+-----------+---------------+------------------+------------+------------------+-----------------+------------------+---------------+------------------+-----------------+------------------+
|      Jake Delhomme|  2006|     26078|     2006.0|            263| 20.23076923076923|         431| 33.15384615384615|           2805.0|215.76923076923077|             17|215.76923076923077|             11.0|0.8461538461538461|
|       Jake Plummer|  2005|     32080|     2005.0|            277|           17.3125|  

#### Create two new variables

- Create two new variables (by season/player combination):\
∗ completion_percentage = (sum of completions)/(sum of attempts)\
∗ td_int_ratio = (sum passing tds)/(sum interceptions)

In [59]:
qb_sub_new = (qb_sub_summary.withColumn("completion_percentage", F.try_divide("completions_sum", "attempts_sum"))
                            .withColumn("td_int_ratio", F.try_divide("passing_tds_sum", "interceptions_sum")))
qb_sub_new.limit(5).toPandas().round(3)

,player_display_name,season,season_sum,season_mean,completions_sum,completions_mean,attempts_sum,attempts_mean,passing_yards_sum,passing_yards_mean,passing_tds_sum,passing_yards_mean,interceptions_sum,interceptions_mean,completion_percentage,td_int_ratio
0,Jake Delhomme,2006,26078,2006.0,263,20.231,431,33.154,2805.0,215.769,17,215.769,11.0,0.846,0.610,1.545
1,Jake Plummer,2005,32080,2005.0,277,17.312,456,28.500,3366.0,210.375,18,210.375,7.0,0.438,0.607,2.571
2,Matt Schaub,2006,10030,2006.0,18,3.600,27,5.400,208.0,41.600,1,41.600,2.0,0.400,0.667,0.500
3,Vince Young,2006,30090,2006.0,184,12.267,356,23.733,2199.0,146.600,12,146.600,13.0,0.867,0.517,0.923
4,Kerry Collins,2007,12042,2007.0,50,8.333,82,13.667,531.0,88.500,0,88.500,0.0,0.000,0.610,NaN


This output shows that the new variables `completion_percentage` and `td_int_ratio` were successfully created and appended to the right side of the DataFrame, with their corresponding values calculated. The results were saved to the object `qb_sub_new`.

#### Subset and sort data

- Now, with that object qb_sub_new, we will\
    – Subset the rows to only include player/season combinations where the sum of attempts is at least 50.\
    – Sort the rows descending by completion_percentage and report the first 40 values!\
    – Sort the rows descending by td_int_ratio and report the first 40 values!

In [66]:
# Subset the rows to only include player/season combinations where the sum of attempts is at least 50.
qb_sub_new_filter = qb_sub_new.filter(F.col("attempts_sum") > 50)

# sort the rows desencing by completion_percentage and report the first 40 values!
qb_sub_new_sort = qb_sub_new_filter.orderBy(F.col("completion_percentage").desc())

# show the first 40 values in pandas style
qb_sub_new_sort.limit(40).toPandas().round(3)

,player_display_name,season,season_sum,season_mean,completions_sum,completions_mean,attempts_sum,attempts_mean,passing_yards_sum,passing_yards_mean,passing_tds_sum,passing_yards_mean,interceptions_sum,interceptions_mean,completion_percentage,td_int_ratio
0,C.J. Beathard,2023,12138,2023.0,40,6.667,53,8.833,349.0,58.167,1,58.167,0.0,0.000,0.755,NaN
1,Colt McCoy,2021,14147,2021.0,74,10.571,99,14.143,740.0,105.714,3,105.714,1.0,0.143,0.747,3.000
2,Matt Schaub,2019,10095,2019.0,50,10.000,67,13.400,580.0,116.000,3,116.000,1.0,0.200,0.746,3.000
3,Drew Brees,2018,30270,2018.0,364,24.267,489,32.600,3992.0,266.133,32,266.133,5.0,0.333,0.744,6.400
4,Drew Brees,2019,22209,2019.0,281,25.545,378,34.364,2979.0,270.818,27,270.818,4.0,0.364,0.743,6.750
5,Mason Rudolph,2023,8092,2023.0,55,13.750,74,18.500,719.0,179.750,3,179.750,0.0,0.000,0.743,NaN
6,Taysom Hill,2020,32320,2020.0,88,5.500,121,7.562,928.0,58.000,4,58.000,2.0,0.125,0.727,2.000
7,Nick Foles,2018,10090,2018.0,141,28.200,195,39.000,1413.0,282.600,7,282.600,4.0,0.800,0.723,1.750
8,Drew Brees,2017,32272,2017.0,386,24.125,536,33.500,4334.0,270.875,23,270.875,8.0,0.500,0.720,2.875
9,Sam Bradford,2016,30240,2016.0,395,26.333,552,36.800,3877.0,258.467,20,258.467,5.0,0.333,0.716,4.000


The output shows that the DataFrame has been sorted descending by `completion_percentage` and the 40 values are dispalyed.

- Now, sort the rows descending by `td_int_ratio` and report the first 40 values!

In [68]:
# sort the rows desencing by completion_percentage and report the first 40 values!
qb_sub_new_sort = qb_sub_new_filter.orderBy(F.col("td_int_ratio").desc())

# show the first 40 values in pandas style
qb_sub_new_sort.limit(40).toPandas().round(3)

,player_display_name,season,season_sum,season_mean,completions_sum,completions_mean,attempts_sum,attempts_mean,passing_yards_sum,passing_yards_mean,passing_tds_sum,passing_yards_mean,interceptions_sum,interceptions_mean,completion_percentage,td_int_ratio
0,Tom Brady,2016,24192,2016.0,291,24.250,432,36.000,3554.0,296.167,28,296.167,2.0,0.167,0.674,14.000
1,Nick Foles,2013,26169,2013.0,203,15.615,317,24.385,2891.0,222.385,27,222.385,2.0,0.154,0.640,13.500
2,Josh McCown,2013,16104,2013.0,149,18.625,224,28.000,1829.0,228.625,13,228.625,1.0,0.125,0.665,13.000
3,Aaron Rodgers,2018,32288,2018.0,372,23.250,597,37.312,4442.0,277.625,25,277.625,2.0,0.125,0.623,12.500
4,Damon Huard,2006,20060,2006.0,148,14.800,244,24.400,1878.0,187.800,11,187.800,1.0,0.100,0.607,11.000
5,Aaron Rodgers,2020,32320,2020.0,372,23.250,526,32.875,4299.0,268.688,48,268.688,5.0,0.312,0.707,9.600
6,Aaron Rodgers,2021,32336,2021.0,366,22.875,531,33.188,4115.0,257.188,37,257.188,4.0,0.250,0.689,9.250
7,Tom Brady,2010,32160,2010.0,324,20.250,492,30.750,3900.0,243.750,36,243.750,4.0,0.250,0.659,9.000
8,Jake Delhomme,2007,6021,2007.0,55,18.333,86,28.667,617.0,205.667,8,205.667,1.0,0.333,0.640,8.000
9,Aaron Rodgers,2014,32224,2014.0,341,21.312,520,32.500,4381.0,273.812,38,273.812,5.0,0.312,0.656,7.600


The output shows the DataFrame sorted in descending order by `td_int_ratio`, and the first 40 values are displayed. Rows with missing values do not appear at the top because Spark SQL treats `NULL` values as smaller than numeric values when sorting in descending order, which differs from the behavior of pandas‑on‑Spark.

## Brief summary of Part II
In Part II, I conducted a basic exploratory analysis of the NFL weekly dataset using both `pandas`‑on‑Spark and Spark `SQL`. I loaded the data, subset the rows and columns to focus on quarterback statistics, and computed season‑level sums and means grouped by player_display_name and season. I then created two derived variables, completion percentage and touchdown‑to‑interception ratio, and filtered the results to players with at least 50 passing attempts. Finally, I sorted the data by each derived metric and examined the top values.

Throughout the analysis, I compared the behavior of pandas‑on‑Spark and `Spark SQL`, noting differences in syntax, column creation, error handling, and sorting behavior for missing values. Both methods produced consistent results, but Spark SQL required more explicit transformations and followed SQL semantics for `NULL` handling.